# 🧬 ProteinMind Agent — Colab Notebook
### Agentic Protein Methylation Prediction | HybridARGME Multi-Agent Framework
**M.Tech Thesis | Dataset-Centric | No External API Calls**

---
**Steps:**
1. Mount Google Drive
2. Install dependencies
3. Set paths
4. Run the full pipeline
5. Download your PDF report

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

## Cell 2 — Install Dependencies

In [ ]:
%%capture install_output
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install fair-esm==2.0.0
!pip install reportlab>=4.0.0
!pip install matplotlib seaborn scikit-learn pandas numpy joblib requests biopython tqdm
print('All packages installed!')

In [ ]:
# Verify key imports
import torch
import reportlab
import matplotlib
print(f'torch      : {torch.__version__}')
print(f'reportlab  : {reportlab.Version}')
print(f'matplotlib : {matplotlib.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## Cell 3 — Configure Paths
> **Edit `DRIVE_ROOT`** if your folder is in a different location on Drive.

In [ ]:
import sys, os

# ============================================================
#  EDIT THIS — path where you uploaded ProteinMind_Agent
# ============================================================
DRIVE_ROOT = "/content/drive/MyDrive/ProteinMind_Agent"
# ============================================================

DATA_DIR    = os.path.join(DRIVE_ROOT, "data-11")
OUTPUT_DIR  = os.path.join(DRIVE_ROOT, "proteinmind_outputs")
CHECKPOINT  = os.path.join(DRIVE_ROOT, "checkpoints", "pretrained_esm_v4.pt")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Add project root to Python path
if DRIVE_ROOT not in sys.path:
    sys.path.insert(0, DRIVE_ROOT)
os.chdir(DRIVE_ROOT)

# Verify
print('Project root :', DRIVE_ROOT)
print('Data dir     :', DATA_DIR,   '| exists:', os.path.exists(DATA_DIR))
print('Checkpoint   :', CHECKPOINT, '| exists:', os.path.exists(CHECKPOINT))
print('Output dir   :', OUTPUT_DIR)
print()

if not os.path.exists(DATA_DIR):
    print('[WARNING] data-11 folder not found! Check your DRIVE_ROOT path.')
else:
    # Count CSV files in data-11
    csv_count = sum(1 for r,d,f in os.walk(DATA_DIR) for fn in f if fn.endswith('.csv'))
    print(f'Found {csv_count} CSV files in data-11  [OK]')

## Cell 4 — Load All Agents (One-time Init)
> This loads dataset into memory (~250K entries). Run once per session.

In [ ]:
from agents.orchestrator import Orchestrator
from visuals.generator import VisualsGenerator
from report.clinical_report import ClinicalReportGenerator

print('Initializing ProteinMind Orchestrator...')
orch = Orchestrator(
    checkpoint_path=CHECKPOINT if os.path.exists(CHECKPOINT) else None,
    data_dir=DATA_DIR
)
print('\nOrchestrator ready!')

## Cell 5 — Run Analysis
> **Edit the query below.** Supported formats:
> - `Analyze sequence: MKRGGKPLRTA...` (full protein)
> - `Analyze motif: XXXRGGRVLSX` (11-mer)
> - `Analyze sample #125 from dataset` (dataset row lookup)

In [ ]:
# ============================================================
#  EDIT YOUR QUERY HERE
# ============================================================
QUERY = "Analyze sequence: MKRGGKPLRTAMKRGGKPLRTASRGGRGGFGSRGGRGGMKRTAM"

# Optional: add real protein metadata for the PDF report
METADATA = {
    "protein_name": "Query Protein",     # e.g. "hnRNP A1"
    "gene_name"   : "N/A",               # e.g. "HNRNPA1"
    "organism"    : "Homo sapiens",
    "function"    : "Dataset-centric methylation prediction via HybridARGME.",
}
# ============================================================

print('Running analysis...')
print(f'Query: {QUERY}')
print()

result = orch.execute_analysis(QUERY)

if not result.get('success'):
    print(f'[ERROR] {result.get("error")}')
else:
    result['metadata']   = METADATA
    result['uniprot_id'] = 'N/A'

    summary  = result.get('predictions_summary', {})
    positives = result.get('positive_predictions', [])

    print('=' * 60)
    print('RESULTS SUMMARY')
    print('=' * 60)
    print(f"Query ID       : {result.get('query_id')}")
    print(f"Sequence length: {len(result.get('input_sequence', ''))} aa")
    print(f"Sites evaluated: {summary.get('total_sites_checked')}")
    print(f"Positive sites : {summary.get('positive_sites_count')}")
    print(f"Threshold      : {summary.get('threshold_used')}")
    print(f"Fusion mode    : {summary.get('fusion_mode')}")
    print()

    if positives:
        print('Predicted Methylation Sites:')
        for p in positives:
            print(f"  {p['residue']}{p['position']:4d}  "
                  f"Motif: {p['motif']}  "
                  f"P_final={p.get('P_final',0):.4f}  "
                  f"[{p.get('confidence','N/A')}]  "
                  f"Evidence={p.get('evidence_tier','N/A')}")
    else:
        print('No sites predicted above threshold.')

    print('\nAnalysis complete!')

## Cell 6 — Generate All Visualizations

In [ ]:
if result.get('success'):
    print(f'Generating figures -> {OUTPUT_DIR}')
    gen     = VisualsGenerator(output_dir=OUTPUT_DIR)
    visuals = gen.execute_all(result)

    print('\nFigures generated:')
    for key, val in visuals.items():
        if isinstance(val, list):
            for i, p in enumerate(val):
                if p and os.path.exists(p):
                    print(f'  [{key}][{i+1}] {os.path.basename(p)}')
        elif val and os.path.exists(val):
            print(f'  [{key}] {os.path.basename(val)}')
else:
    print('Skip — no successful result.')

## Cell 7 — Preview Figures in Notebook

In [ ]:
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show_figure(path, title=''):
    if path and os.path.exists(path):
        print(f'--- {title} ---')
        img = mpimg.imread(path)
        plt.figure(figsize=(14, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=12, fontweight='bold', color='#1e3a8a')
        plt.tight_layout()
        plt.show()
    else:
        print(f'[SKIP] {title} — file not found')

if result.get('success'):
    show_figure(visuals.get('structure_3d',''),  '3D Protein Backbone Structure')
    show_figure(visuals.get('structure_2d',''),  '2D Secondary Structure Map')
    show_figure(visuals.get('sequence_annotation',''), 'Sequence Methylation Site Map')
    show_figure(visuals.get('agent_score_heatmap',''), 'Multi-Agent Score Heatmap')

    for i, radar in enumerate(visuals.get('radar_charts', [])):
        show_figure(radar, f'Agent Radar Chart — Site {i+1}')

    for i, card in enumerate(visuals.get('site_cards', [])):
        show_figure(card, f'Site Detail Card {i+1}')

    for i, attn in enumerate(visuals.get('attention_maps', [])):
        show_figure(attn, f'BMAM Attention Map — Site {i+1}')

    for i, shap in enumerate(visuals.get('shap_plots', [])):
        show_figure(shap, f'SHAP Attribution — Site {i+1}')

## Cell 8 — Generate PDF Report

In [ ]:
if result.get('success'):
    query_id = result.get('query_id', 'query')
    pdf_path = os.path.join(OUTPUT_DIR, f'report_{query_id}.pdf')

    print(f'Building PDF report -> {pdf_path}')
    rep = ClinicalReportGenerator(pdf_path)
    rep.generate_pdf(result, visuals)

    size_kb = os.path.getsize(pdf_path) / 1024
    print(f'\nPDF generated: {os.path.basename(pdf_path)}  ({size_kb:.1f} KB)')
    print(f'Full path: {pdf_path}')
else:
    print('Skip — no successful result.')

## Cell 9 — Download PDF

In [ ]:
from google.colab import files

if result.get('success') and os.path.exists(pdf_path):
    print(f'Downloading: {os.path.basename(pdf_path)}')
    files.download(pdf_path)
    print('Download started!')
else:
    print('PDF not found — check Cell 8.')

## Cell 10 — (Optional) Open 3D Viewer in Browser Tab

In [ ]:
# Displays the interactive 3D molecular viewer inline in Colab
from IPython.display import IFrame

viewer_path = visuals.get('3d_viewer', '')
if viewer_path and os.path.exists(viewer_path):
    # Read HTML content and display inline
    with open(viewer_path, 'r', encoding='utf-8') as f:
        html_content = f.read()
    from IPython.display import HTML
    display(HTML(html_content))
else:
    print('3D HTML viewer not found.')

## Cell 11 — (Optional) Run CLI Command Instead

In [ ]:
# Alternative: run the full pipeline via CLI in one command
# Edit the --query flag below
!python run_proteinmind.py \
    --query "Analyze sequence: MKRGGKPLRTAMKRGGKPLRTASRGGRGGFGSRGGRGG" \
    --drive "/content/drive/MyDrive/ProteinMind_Agent" \
    --data-dir "data-11" \
    --checkpoint "checkpoints/pretrained_esm_v4.pt" \
    --output-dir "proteinmind_outputs"

---
## Notes
- **GPU**: Enable via Runtime → Change runtime type → T4 GPU
- **Drive path**: Make sure `ProteinMind_Agent/` is at `MyDrive/ProteinMind_Agent/`
- **Checkpoint**: If `pretrained_esm_v4.pt` is missing, the system runs in demo mode (untrained weights). Predictions will still be generated.
- **PDF**: Saved to `proteinmind_outputs/report_<query_id>.pdf` on your Drive
- **Dataset**: 250,426 entries from Human, Mouse, Rat, Chimpanzee methylation CSVs in `data-11/`